In [84]:
# load all the files under the HRUKBB dire
import os
import sys 
sys.path.append(os.path.join('..'))


import numpy as np
import torch
import torch.nn.functional as F
import data.data_utils as dut
from data_process.dataset_real_scaling import *
from GHD.GHD_cardiac import GHD_Cardiac
from GHD import GHD_config
from ops.medical_related import get_4chamberview_frame
from pytorch3d.transforms import axis_angle_to_matrix, matrix_to_axis_angle
# from pytorch3d.io import save_obj
import trimesh
# load the mesh, slices 
import os
import sys 
sys.path.append(os.path.join('..'))


import nibabel as nib

from pytorch3d.ops import sample_points_from_meshes, cubify

import pyvista as pv
pv.start_xvfb(wait=0)
pv.set_jupyter_backend('html')


import data.data_utils as dut
from torch_scatter import scatter
from einops import rearrange
from ops.medical_related import *


In [85]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

datadir = '/media/ssd/fanwen/MultiView/HRUKBB/Dataset'
rootdir = '/media/ssd/fanwen/MultiView/HRUKBB/'
savedir = '/media/ssd/fanwen/MultiView/HRUKBB/GHBMesh'

root_path = os.path.dirname(os.path.realpath('.'))
base_shape_path = 'canonical_shapes/Standard_LV_2000.obj'
base_shape_path = os.path.join(root_path, base_shape_path)
bi_ventricle_path = 'canonical_shapes/Standard_BiV.obj'
bi_ventricle_path = os.path.join(root_path, bi_ventricle_path)

loss_dict = {'Loss_occupancy':1., 'Loss_normal_consistency':0.01, 'Loss_Laplacian':0.1, 'Loss_thickness':0.02}

# base_shape_path = 'metadata/Standard_LV.obj'
# bi_ventricle_path = 'metadata/Standard_BiV.obj'

cfg = GHD_config(base_shape_path=base_shape_path,
            num_basis=6**2, mix_laplacian_tradeoff={'cotlap':1.0, 'dislap':0.1, 'stdlap':0.1},
            device='cuda:0',
            if_nomalize=True, if_return_scipy=True, 
            bi_ventricle_path=bi_ventricle_path)

paraheart = GHD_Cardiac(cfg) #

casename = '1138'
# casename = '1128'

casedir = os.path.join(datadir, casename)
timing = 'ES'
sample_num = 2000
print(casedir)

GHD config:
base_shape_path /home/fanwen/GHDHeart/canonical_shapes/Standard_LV_2000.obj
num_basis 36
device cuda:0
mix_laplacian_tradeoff {'cotlap': 1.0, 'dislap': 0.1, 'stdlap': 0.1}
if_lap_nomalize True
eign_path None
if_nomalize True
if_return_scipy True
bi_ventricle_path /home/fanwen/GHDHeart/canonical_shapes/Standard_BiV.obj
/media/ssd/fanwen/MultiView/HRUKBB/Dataset/1138


In [86]:
def affine_np2torch(affine_np, 
                    img_size_np, 
                    rescalar = 1/100, 
                    center_aligned = True):
    '''
    convert affine matrix from numpy manner to torch manner (normed real world)
    affine_np: [4, 4] original affine matrix read from the medical image file
    img_size: [3] the size of the image, [x, y, z]
    rescalar: [1] the rescalar of the image, default is 1/100mm, [-100mm, 100mm] -> [-1, 1]
    center_aligned: [bool] whether the image is center aligned, default is True
    '''
    if isinstance(affine_np, np.ndarray):
        affine_np2w = torch.from_numpy(affine_np).float()
    else:
        affine_np2w = affine_np.float()

    affine_torch2np = torch.diag(torch.tensor([img_size_np[0]-1., 
                                               img_size_np[1]-1., 
                                               img_size_np[2]-1., 1.]))/2.
    if center_aligned:
        affine_torch2np[:3, 3] = 0 # set the translation to 0
        # revised by fanwen
        # affine_np2w[:3,3] = 0
    else:
        affine_torch2np[:3, 3] = torch.tensor([img_size_np[0]-1., 
                                               img_size_np[1]-1., 
                                               img_size_np[2]-1.])/2.
    affine_t2w_ =  affine_np2w @ affine_torch2np
    affine_t2w_[:3,:] = affine_t2w_[:3,:]*rescalar
    affine_t2w_[3,:] = torch.tensor([0,0,0,1])
    return affine_t2w_

def affine2paras(affine):
    # the scales should be the norm of the rotation matrix, not skipping the real part. 
    '''
    transform the afffine to translation, rotations and scales
    Args:
        affine: (n,4,4)
    Return:
        translation: (n,3)
        rotation: (n,3)
        scales: (n,3)   
    '''
    translation = affine[:,:3,3]
    scales = torch.norm(affine[:,:3,:3], dim = 1)
    # norm the rotation matrix
    rotats = affine[:,:3,:3]/torch.norm(affine[:,:3,:3], dim=2, keepdim=True)
    rotation = matrix_to_axis_angle(rotats)
    return translation, rotation, scales

In [87]:
def affine_npimgzyx2torch(affine_np, 
                    label_torch_size, 
                    rescalar = 1/100, 
                    center_aligned = True):
    '''
    convert affine matrix from numpy manner to torch manner (normed real world)
    affine_np: [4, 4] original affine matrix read from the medical image file
    img_size: [3] the size of the image, [z, y, x]
    rescalar: [1] the rescalar of the image, default is 1/100mm, [-100mm, 100mm] -> [-1, 1]
    center_aligned: [bool] whether the image is center aligned, default is True
    '''
    if isinstance(affine_np, np.ndarray):
        affine_np2w = torch.from_numpy(affine_np).float()
    else:
        affine_np2w = affine_np.float()

    affine_torch2np = torch.diag(torch.tensor([label_torch_size[-1]-1., 
                                               label_torch_size[-2]-1., 
                                               label_torch_size[-3]-1., 
                                               1.]))/2.
    if center_aligned:
        affine_np2w[:3, 3] = 0 # set the translation to 0
    else:
        affine_torch2np[:3, 3] = torch.tensor([label_torch_size[-1]-1., 
                                               label_torch_size[-2]-1., 
                                               label_torch_size[-3]-1.])/2.
    
    affine_t2w_ =  affine_np2w @ affine_torch2np
    affine_t2w_[:3,:] = affine_t2w_[:3,:]*rescalar
    affine_t2w_[3,:] = torch.tensor([0,0,0,1])
    return affine_t2w_

In [88]:
def get_4chamberview_frame_noupsidedown(cav_pts, lv_pts, rv_pts):
    '''find the frame for 4 chamber view, from points smapled from 3 cavity&ventricle.
    cav_pts: torch.tensor, (N, 3)
    lv_pts: torch.tensor, (N, 3)
    rv_pts: torch.tensor, (N, 3)

    return: target_affine: torch.tensor, (4, 4),

    '''
    Pt_center = cav_pts.mean(dim=0) # center of the cavity
    Pt_center_lv = lv_pts.mean(dim=0) # center of the lv
    Pt_center_rv = rv_pts.mean(dim=0) # center of the rv

    Cov_cav = torch.matmul((cav_pts-Pt_center).transpose(0,1), cav_pts-Pt_center)/cav_pts.shape[-2]
    
    _, eigvecs = torch.linalg.eigh(Cov_cav)

    u2d_axis = eigvecs[:,-1]


    u2d_axis_proj = (lv_pts-Pt_center).matmul(u2d_axis)
    
    u2d_axis_proj_max, u2d_axis_proj_max_idx = u2d_axis_proj.max(dim=0)
    u2d_axis_proj_min, u2d_axis_proj_min_idx = u2d_axis_proj.min(dim=0)

    dist_max = (lv_pts[u2d_axis_proj_max_idx]-Pt_center) - u2d_axis_proj_max*u2d_axis
    dist_min = (lv_pts[u2d_axis_proj_min_idx]-Pt_center) - u2d_axis_proj_min*u2d_axis


    l2r_axis = Pt_center_rv-Pt_center
    l2r_axis = l2r_axis/torch.norm(l2r_axis)
    b2f_axis = torch.cross(l2r_axis, u2d_axis)
    b2f_axis = b2f_axis/torch.norm(b2f_axis)
    l2r_axis = torch.cross(u2d_axis, b2f_axis)

    target_frame = torch.stack([-b2f_axis, l2r_axis, -u2d_axis], dim=1)

    target_affine = torch.eye(4).to(cav_pts.device)
    target_affine[:3,:3] = target_frame 
    target_affine[:3,3] = Pt_center

    return {'mean_cav':Pt_center, 'mean_lv':Pt_center_lv, 'mean_rv':Pt_center_rv, 
            'b2f_axis':b2f_axis, 'l2r_axis':l2r_axis, 'u2d_axis':u2d_axis,
            'target_affine':target_affine}

In [89]:
hr_nii_path = os.path.join(casedir, 'HR_{}.nii.gz'.format(timing))
output = dut.load_nib_image(hr_nii_path)

labels = torch.Tensor(output['img']).unsqueeze(0).unsqueeze(0)

label_torch = torch.from_numpy(output['img']).permute(2,1,0).unsqueeze(0).float()
affine_torch = torch.from_numpy(output['affine']).float()
rescalar = 1/100
# rescalar = 1/100

affine_t2w_ = affine_npimgzyx2torch(output['affine'],
                                    label_torch.shape[-3:], 
                                    rescalar=rescalar, 
                                    center_aligned=True)

# affine_t2w_ = affine_np2torch(output['affine'],
#                             output['img'].shape[-3:], 
#                             rescalar=1/100, 
#                             center_aligned=True)

label_tem = label_torch.squeeze(0)
# label_tem = label_img
Z, Y, X = label_tem.shape

coordinate_map = dut.get_coord_map_3d_normalized(label_tem.shape[-3:], 
                                                affine_t2w_)

Z_rv, Y_rv, X_rv = torch.where(label_tem==4)
Z_lv, Y_lv, X_lv = torch.where(label_tem==2)
Z_cav, Y_cav, X_cav = torch.where(label_tem==1)
Z_bg, Y_bg, X_bg = torch.where(label_tem==0)


Pt_rv = coordinate_map[0, Z_rv, Y_rv, X_rv]
Pt_lv = coordinate_map[0, Z_lv, Y_lv, X_lv]
Pt_cav = coordinate_map[0, Z_cav, Y_cav, X_cav]
Pt_bg = coordinate_map[0, Z_bg, Y_bg, X_bg]

points_bi = torch.cat([Pt_rv, Pt_lv], dim=0)
points_lv = Pt_lv
points_outoflv = torch.cat([Pt_rv, Pt_cav, Pt_bg], dim=0)

# why 
geom_dict = get_4chamberview_frame(Pt_cav, Pt_lv, Pt_rv)
inital_affine = geom_dict['target_affine']

bbox_lv = torch.stack([Pt_lv.min(dim=0)[0]-0.05, Pt_lv.max(dim=0)[0]+0.05], dim=-1)

points_outoflv_in_bbox = points_outoflv[(points_outoflv[:,0]>bbox_lv[0,0]) & (points_outoflv[:,0]<bbox_lv[0,1]) & (points_outoflv[:,1]>bbox_lv[1,0]) & (points_outoflv[:,1]<bbox_lv[1,1]) & (points_outoflv[:,2]>bbox_lv[2,0]) & (points_outoflv[:,2]<bbox_lv[2,1])]

paraheart.R = matrix_to_axis_angle(inital_affine[...,:3,:3].to(paraheart.device)).view(paraheart.R.shape)
paraheart.T = inital_affine[...,:3,3].to(paraheart.device).view(paraheart.T.shape)

In [90]:
mesh_gt_bi_sample = points_bi.detach().cpu().numpy()[np.random.choice(points_bi.shape[0], sample_num, replace=False)]
paraheart.global_registration_biv(mesh_gt_bi_sample)
bi_paraheart = paraheart

In [91]:
label_tem = label_tem.unsqueeze(0).unsqueeze(0).to(device)

In [92]:
coordinate_map_np = coordinate_map[0].detach().cpu().numpy()

pl = pv.Plotter(notebook=True)
interval = 1
if label_tem.shape[-3] > 20:
    interval = 2
if label_tem.shape[-3] > 50:
    interval = 10
for i in range(0, label_tem.shape[-3], interval):

    x, y, z = coordinate_map_np[i,...,0], coordinate_map_np[i,...,1], coordinate_map_np[i,...,2]

    grid = pv.StructuredGrid(x, y, z)

    color_gt = (label_tem[0,0,i].cpu().numpy().T.flatten() ==2).astype(np.float32)
    
    raw_image = label_tem[0,0,i].cpu().numpy().T.flatten()

    color_opacity = np.ones_like(color_gt)*0.8

    color_opacity[color_gt == 0] = 0.2

    color_gt = raw_image*(1-color_gt) + color_gt

    pl.add_mesh(grid, scalars = color_gt, cmap = 'gray_r',
                show_scalar_bar = False, opacity = color_opacity, clim=[0,1])
    
out_ghd_mesh = bi_paraheart.rendering()

# trimesh_current_bi = paraheart.rendering_bi_ventricle()
# trimesh_current_bi = pv.wrap(trimesh_current_bi)
# pl.add_mesh(trimesh_current_bi, color='blue', opacity=0.2)

trimesh_current_lv = trimesh.Trimesh(out_ghd_mesh.verts_packed().detach().cpu().numpy(), 
out_ghd_mesh.faces_packed().detach().cpu().numpy())
pl.add_mesh(trimesh_current_lv, color='lightblue', opacity=0.8, show_edges=True, show_vertices=False)


pl.add_mesh(pv.Box(bounds=[-1, 1, -1, 1, -1, 1]).outline(), color='black')

# pl.add_points(lv_cavity_center.cpu().numpy(), color='red', point_size=10)

pl.add_axes()
pl.show()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

In [93]:
sample_lv = points_lv[np.random.choice(points_lv.shape[0], sample_num, replace=False)]
sample_outoflv = points_outoflv_in_bbox[np.random.choice(points_outoflv_in_bbox.shape[0], sample_num, replace=False)]
paraheart.global_registration_lv(sample_lv.detach().cpu().numpy())
lv_paraheart = paraheart

In [94]:
coordinate_map_np = coordinate_map[0].detach().cpu().numpy()

pl = pv.Plotter(notebook=True)
interval = 1
if label_tem.shape[-3] > 20:
    interval = 2
if label_tem.shape[-3] > 50:
    interval = 10
for i in range(0, label_tem.shape[-3], interval):

    x, y, z = coordinate_map_np[i,...,0], coordinate_map_np[i,...,1], coordinate_map_np[i,...,2]

    grid = pv.StructuredGrid(x, y, z)

    color_gt = (label_tem[0,0,i].cpu().numpy().T.flatten() ==2).astype(np.float32)
    
    raw_image = label_tem[0,0,i].cpu().numpy().T.flatten()

    color_opacity = np.ones_like(color_gt)*0.8

    color_opacity[color_gt == 0] = 0.2

    color_gt = raw_image*(1-color_gt) + color_gt

    pl.add_mesh(grid, scalars = color_gt, cmap = 'gray_r',
                show_scalar_bar = False, opacity = color_opacity, clim=[0,1])
    
out_ghd_mesh = lv_paraheart.rendering()

# trimesh_current_bi = paraheart.rendering_bi_ventricle()
# trimesh_current_bi = pv.wrap(trimesh_current_bi)
# pl.add_mesh(trimesh_current_bi, color='blue', opacity=0.2)

trimesh_current_lv = trimesh.Trimesh(out_ghd_mesh.verts_packed().detach().cpu().numpy(), 
out_ghd_mesh.faces_packed().detach().cpu().numpy())
pl.add_mesh(trimesh_current_lv, color='lightblue', opacity=0.8, show_edges=True, show_vertices=False)


pl.add_mesh(pv.Box(bounds=[-1, 1, -1, 1, -1, 1]).outline(), color='black')

# pl.add_points(lv_cavity_center.cpu().numpy(), color='red', point_size=10)

pl.add_axes()
pl.show()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

In [95]:
# sample_outoflv = points_outoflv_in_bbox[np.random.choice(points_outoflv_in_bbox.shape[0], sample_num*5, replace=False)]
convergence, Loss_dict_list  = paraheart.morphing2lvtarget(points_lv.to(device), 
                                                            points_outoflv_in_bbox.to(device), 
                                                            loss_dict = {'Loss_occupancy':1, 'Loss_Laplacian':0.001, 'Loss_thickness': 0.001},
                                                            lr_start=1e-3, 
                                                            num_iter=2000, 
                                                            if_reset=True, 
                                                            if_fit_R=False, 
                                                            if_fit_s=True, 
                                                            if_fit_T=True, 
                                                            record_convergence=True)

Total Loss 0.0545: 100%|██████████| 2000/2000 [01:15<00:00, 26.34it/s]

fittings done, the final loss is 0.054451


In [96]:
rotation = paraheart.R.detach().cpu()
translation = paraheart.T.detach().cpu()

R = axis_angle_to_matrix(rotation).view(3,3)
T = translation.view(3,1)

# here is the affine matrix of the paraheart transformation
affine_c2p = torch.eye(4)
affine_c2p[:3,:3] = R
affine_c2p[:3,3] = T.squeeze()
affine_c2p[3, 3] = 1.0

In [97]:
affine_in = affine_c2p.inverse()

paraheart.R = torch.Tensor([0, 0, 0]).view(1,3).to(paraheart.device)
paraheart.T = torch.Tensor([0, 0, 0]).view(1,3).to(paraheart.device)

current_mesh = paraheart.rendering()

In [98]:
def get_2D_meshgrid_norm_world(affine_t: torch.Tensor, # 4*4
                               matrix_size: int = 128, 
                               spacing: float = 1.8,
                               ):
    '''
    get the 2D meshgrid in the normed real world
    affine_t2w_: [4, 4] the affine matrix from numpy to torch
    rotate_ma: [3, 3] the rotation matrix
    meshgrid2D: [128, 128, 3] the meshgrid in the normed real world
    center_aligned: [bool] whether the image is center aligned, default is True
    '''
    norm_window_size = (matrix_size - 1)*spacing/200
    x, y = torch.meshgrid(torch.linspace(-norm_window_size, norm_window_size, matrix_size), 
                          torch.linspace(-norm_window_size, norm_window_size, matrix_size))
    z = torch.zeros_like(x)
    meshgrid2D = torch.stack([x, y, z], dim=-1)
    meshgrid2D_ = meshgrid2D @ affine_t[:3, :3].transpose(0,1) + affine_t[:3, -1]
    return meshgrid2D_

def sample_slices(label_tem: torch.Tensor, 
                  meshgrid2D: torch.Tensor, 
                  affine_t2w_: torch.Tensor):
    '''
    get the 5D grid for the image
    label_tem: [B, C, Z, Y, X] the 3D image to
    meshgrid2D: the meshgrid with physical sampling in the normed real world.
    return: img_size: [1, 1, X, Y, 1]
    '''
    if len(label_tem.shape) ==3:
        label_tem = label_tem.unsqueeze(0).unsqueeze(0)
    elif len(label_tem.shape) == 4:
        label_tem = label_tem.unsqueeze(0)

    affine_t2w_i = affine_t2w_.inverse()
    meshgrid2D = torch.Tensor(meshgrid2D) # meshgrid in x,y,z
    mesh_grid_t2s = meshgrid2D @ affine_t2w_i[:3, :3].T + affine_t2w_i[:3, 3]
    # change this to a y,x,z sequence.
    grid_5D = mesh_grid_t2s.permute(1,0,2).unsqueeze(0).unsqueeze(0)
    image = F.grid_sample(label_tem, grid_5D) # B, C, 1, 128, 128. (BCZYX)
    return image.squeeze(0).squeeze(0).permute(2,1,0)

In [99]:
# slicing the HR image stacks. 

# sax 
rotate_sax = torch.stack((
    geom_dict['l2r_axis'],
    geom_dict['b2f_axis'],
    -geom_dict['u2d_axis']
), dim=1) 

# 
affines_slice = []
# generate a series of slices for SAX 
for i in range(-5,4,1):
    affine_sax = torch.eye(4)
    affine_sax[2,3] += i*8/100
    affine_sax[:3, :3] = rotate_sax  # Set the rotation part
    affine_sax[:3, 3] = rotate_sax @ affine_sax[:3,3]  # Set the translation part
    affines_slice.append(affine_sax)

# 4ch view
ch4_rot = torch.stack((
    geom_dict['u2d_axis'],
    geom_dict['l2r_axis'],
    geom_dict['b2f_axis'],
), dim=1)
affine_torch_ch4 = torch.eye(4)
affine_torch_ch4[:3, -1] = geom_dict['mean_cav']
affine_torch_ch4[:3, :3] = ch4_rot
affines_slice.append(affine_torch_ch4)

# 3ch view
degree_3ch = torch.randint(50, 75, (1,)).item()
rotation_4_to_3 = axis_angle_to_matrix(geom_dict['u2d_axis']*torch.pi*(degree_3ch/180))
tangent_3CH = torch.mm(rotation_4_to_3, geom_dict['l2r_axis'].unsqueeze(-1)).squeeze(-1)
normal_3CH = torch.cross(tangent_3CH, geom_dict['u2d_axis'])
rotate_ma_3ch = torch.stack((
    geom_dict['u2d_axis'],
    tangent_3CH,
    normal_3CH,
), dim=1)
affine_torch_3ch = torch.eye(4)
affine_torch_3ch[:3, -1] = geom_dict['mean_cav']
affine_torch_3ch[:3, :3] = rotate_ma_3ch
affines_slice.append(affine_torch_3ch)

# 2ch view
# random number between 110 and 145
degree_2ch = torch.randint(110, 145, (1,)).item()
rotation_4_to_2 = axis_angle_to_matrix(geom_dict['u2d_axis']*torch.pi*(degree_2ch/180))
tangent_2CH = torch.mm(rotation_4_to_2, geom_dict['l2r_axis'].unsqueeze(-1)).squeeze(-1)
normal_2CH = torch.cross(tangent_2CH, geom_dict['u2d_axis'])
rotate_ma_2ch = torch.stack((
    geom_dict['u2d_axis'],
    tangent_2CH,
    normal_2CH,
), dim=1)
affine_torch_2ch = torch.eye(4)
affine_torch_2ch[:3, -1] = geom_dict['mean_cav']
affine_torch_2ch[:3, :3] = rotate_ma_2ch
affines_slice.append(affine_torch_2ch)

# get the affines in the canonical space.
affine_canos, sliced_imgs = [], []
for affine in affines_slice:
    # get the canonical affine. 
    affine_cano = affine_c2p.inverse() @ affine
    affine_canos.append(affine_cano)
    # generate the sliced img. 
    meshgrid_2D = get_2D_meshgrid_norm_world(affine,
                                            128, 
                                            1.8)
    img_slice = sample_slices(label_tem.detach().cpu(),
                            meshgrid_2D,
                            affine_t2w_)
    sliced_imgs.append(img_slice)

affine_canos_matrix = torch.stack(affine_canos, dim=0)
# generate the corrupted affine
sliced_imgs = torch.stack(sliced_imgs, dim=0)

# get the Pv points in the canonical space.
Pt_lv_cano = Pt_lv @ affine_c2p.inverse()[:3, :3].T + affine_c2p.inverse()[:3, 3]
Pt_rv_cano = Pt_rv @ affine_c2p.inverse()[:3, :3].T + affine_c2p.inverse()[:3, 3]
Pt_cav_cano = Pt_cav @ affine_c2p.inverse()[:3, :3].T + affine_c2p.inverse()[:3, 3]

In [100]:
pl = pv.Plotter(notebook=True)

nslice = affine_canos_matrix.shape[0]


for i in range(nslice):
    meshgrid_2D = get_2D_meshgrid_norm_world(affine_canos_matrix[i],
                                            128, 
                                            1.8)
    
    #
    meshgrid_2D_np = meshgrid_2D.detach().cpu().numpy()
    x, y, z = meshgrid_2D_np[...,0], meshgrid_2D_np[...,1], meshgrid_2D_np[...,2]
    grid = pv.StructuredGrid(x, y, z)

    slice_img = sliced_imgs[i]
    
    color_gt = (slice_img[:,:,0].T.detach().cpu().flatten() ==2).numpy().astype(np.float32)
    raw_image = slice_img[:,:,0].T.detach().cpu().flatten()
    
    color_gt = raw_image*(2-color_gt) + color_gt
    
    # color_gt = raw_image

    pl.add_mesh(grid, 
                scalars=color_gt, 
                cmap='gray_r',
                show_scalar_bar=False, 
                opacity=0.4, clim=[1, 3])  # Set opacity to 0.5

pl.add_points(Pt_lv_cano.cpu().numpy(),
             render_points_as_spheres=False, 
             point_size=3, 
             color='blue', opacity=0.1, label='LV')  # Set opacity to 0.5


trimesh_current = trimesh.Trimesh(current_mesh.verts_packed().detach().cpu().numpy(), 
out_ghd_mesh.faces_packed().detach().cpu().numpy())
pl.add_mesh(trimesh_current, color='lightblue', opacity=0.8, show_edges=True, show_vertices=False)

pl.add_mesh(pv.Box(bounds=[-1, 1, -1, 1, -1, 1]).outline(), color='black')
pl.add_axes()

pl.show()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

In [101]:
savedir = '/media/ssd/fanwen/MultiView/HRUKBB/GHBMesh'

HRcase = 'HR_' + timing
save_path = os.path.join(savedir, casename)

output_path = os.path.join(savedir, casename, (HRcase + '_GHD.obj'))

os.makedirs(save_path, exist_ok=True)
# current_mesh.export(os.path.join(save_path, (HRcase + '_GHD.obj')))
# 

# Extract as numpy arrays
verts = current_mesh.verts_packed().detach().cpu().numpy()  # (sum_V, 3)
faces = current_mesh.faces_packed().detach().cpu().numpy()  # (sum_F, 3)

# Build a Trimesh and export in whichever format you like
mesh = trimesh.Trimesh(vertices=verts, faces=faces)
mesh.export(output_path)   # or "mesh.obj", "mesh.stl", ...

'# https://github.com/mikedh/trimesh\nv 0.23742831 0.13974710 0.24879673\nv -0.07715457 0.23974073 0.24325930\nv -0.00998903 0.27007875 0.22494048\nv -0.03785864 0.22696857 0.24111733\nv -0.09597447 0.26341143 0.22703108\nv -0.06577001 0.28841442 0.19733049\nv -0.12686630 0.28042394 0.19915752\nv 0.11056834 0.25694337 0.23914713\nv 0.18397669 0.22490083 0.24215934\nv 0.00675494 0.25524923 0.23650983\nv 0.07277531 0.27109590 0.23081933\nv 0.13110182 0.26245761 0.23644795\nv 0.04117513 0.28735116 0.20719773\nv 0.12678239 0.23531559 0.23920086\nv 0.04594770 0.24588360 0.23644353\nv 0.12664007 0.26971295 0.22685649\nv 0.15852119 0.26235127 0.20594543\nv -0.01267088 0.28903452 0.20402476\nv 0.10005384 0.28107926 0.20737579\nv 0.12977882 0.27975214 0.17471156\nv 0.21837424 0.14005682 0.25112158\nv 0.24849494 0.15787186 0.23423563\nv 0.06097934 0.22142208 0.22678833\nv 0.12294623 0.21621619 0.23367582\nv 0.21327505 0.21081699 0.23165631\nv -0.01215037 0.21618395 0.23121037\nv 0.24777669 0.186

In [103]:
from pytorch3d.io import save_obj

# If you have a batch of meshes, pick one (say the 0th):
verts = current_mesh.verts_list()[0]    # (V, 3) tensor
faces = current_mesh.faces_list()[0]    # (F, 3) tensor

# Make sure they’re on CPU and floats/ints
verts = verts.detach().cpu()
faces = faces.detach().cpu()

# Write
save_obj(output_path, verts, faces)
